In [0]:
%sql

CREATE OR REPLACE TABLE banking.silver.customer_master (
    customer_id INT,
    customer_name STRING,
    city STRING,
    balance DECIMAL(12,2),
    email STRING,
    created_date TIMESTAMP,
    updated_date TIMESTAMP,
    is_active STRING
)
USING DELTA;

In [0]:
%sql
DESCRIBE banking.silver.customer_master;

In [0]:
bronze_df = spark.read.table("banking.bronze.customer_master")

In [0]:
from pyspark.sql.functions import trim, upper, col

silver_df = (
    bronze_df
        .withColumn("cust_name", trim(col("cust_name")))
        .withColumn("city", upper(trim(col("city"))))
)

In [0]:
display(silver_df)

In [0]:
silver_df \
    .withColumnRenamed("cust_name", "customer_name") \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("banking.silver.customer_master")

In [0]:
%sql

SELECT *
FROM banking.silver.customer_master
ORDER BY customer_id;

In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("customer_name", StringType(), False),
    StructField("city", StringType(), True),
    StructField("balance", DecimalType(12,2), True)
])

df = (
    spark.read
         .option("header", "true")
         .schema(schema)
         .csv("/Volumes/banking/bronze/landing_volume/customers_day3_duplicates.csv")
)

In [0]:
display(df)

In [0]:
df.count()

In [0]:
dedup_df = df.dropDuplicates()

In [0]:
dedup_df.count()